# Analysing model performance on video

## Imports

In [5]:
import cv2
import os
from pathlib import Path
import numpy as np
import onnxruntime as ort
import time
from collections import deque, Counter
from ultralytics import YOLO

## Compiling test video

In [6]:
image_folder = Path(r"dataset\PAPI_06\DJI_202604290007_019_300mRwy06night")

# Supported image formats
image_extensions = [".jpg", ".jpeg", ".png"]

# Get images sorted by filename (usually chronological if named by frame/time)
images = sorted([
    img for img in image_folder.iterdir()
    if img.suffix.lower() in image_extensions
])

# Safety check
if not images:
    raise ValueError("No images found in folder")

# Read first image to get size
first_frame = cv2.imread(str(images[0]))
height, width, channels = first_frame.shape

# Video writer (adjust FPS if needed)
fourcc = cv2.VideoWriter_fourcc(*'XVID')
output_video = "output.avi"
fps = 15

video_writer = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

# Write frames
for img_path in images:
    frame = cv2.imread(str(img_path))

    if frame is None:
        print(f"Skipping corrupted image: {img_path}")
        continue

    # Ensure consistent size
    frame = cv2.resize(frame, (width, height))

    video_writer.write(frame)

video_writer.release()

print(f"Video saved to {output_video}")

Video saved to output.avi


## Testing quantized model on video

In [13]:
def letterbox(im, new_shape=640):
    h, w = im.shape[:2]
    scale = min(new_shape / h, new_shape / w)

    nh, nw = int(h * scale), int(w * scale)

    resized = cv2.resize(im, (nw, nh))

    top = (new_shape - nh) // 2
    bottom = new_shape - nh - top
    left = (new_shape - nw) // 2
    right = new_shape - nw - left

    padded = cv2.copyMakeBorder(
        resized, top, bottom, left, right,
        cv2.BORDER_CONSTANT, value=(114, 114, 114)
    )

    return padded, scale, left, top

In [4]:
# -----------------------
# CONFIG
# -----------------------
video_path = "output.avi"          # input video
model_path = "best_int8.onnx"      # your YOLO26n model
output_path = "output_annotated_quantized.avi"

input_size = 640
conf_thres = 0.4

class_names = {
    0: "papi_red",
    1: "papi_white"
}

history = deque(maxlen=5)

# -----------------------
# LOAD MODEL
# -----------------------
session = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])
input_name = session.get_inputs()[0].name

# -----------------------
# VIDEO
# -----------------------
cap = cv2.VideoCapture(video_path)

frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*"XVID")
out = cv2.VideoWriter(output_path, fourcc, fps, (frame_w, frame_h))

# -----------------------
# LETTERBOX (IMPORTANT FOR YOLO)
# -----------------------
def letterbox(im, new_shape=640):
    h, w = im.shape[:2]

    scale = min(new_shape / h, new_shape / w)
    nh, nw = int(h * scale), int(w * scale)

    resized = cv2.resize(im, (nw, nh))

    top = (new_shape - nh) // 2
    bottom = new_shape - nh - top
    left = (new_shape - nw) // 2
    right = new_shape - nw - left

    padded = cv2.copyMakeBorder(
        resized, top, bottom, left, right,
        cv2.BORDER_CONSTANT, value=(114, 114, 114)
    )

    return padded, scale, left, top

# -----------------------
# PREPROCESS
# -----------------------
def preprocess(frame):
    img, scale, pad_x, pad_y = letterbox(frame, input_size)

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.astype(np.float32) / 255.0
    img = np.transpose(img, (2, 0, 1))
    img = np.expand_dims(img, axis=0)

    return img, scale, pad_x, pad_y

# -----------------------
# FPS TRACKING
# -----------------------
prev_time = time.time()

# -----------------------
# MAIN LOOP
# -----------------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    start_time = time.time()

    input_tensor, scale, pad_x, pad_y = preprocess(frame)

    outputs = session.run(None, {input_name: input_tensor})
    preds = outputs[0][0]

    red_count = 0
    white_count = 0
    # -----------------------
    # DETECTIONS
    # -----------------------
    for pred in preds:

        x, y, w, h = pred[:4]
        conf = pred[4]

        if conf < conf_thres:
            continue

        cls_id = int(pred[5]) if len(pred) > 5 else 0
        label = class_names.get(cls_id, str(cls_id))

        # COUNTING LIGHTS
        if cls_id == 0:
            red_count += 1
        elif cls_id == 1:
            white_count += 1

        # -----------------------
        # REMOVE LETTERBOX OFFSET
        # -----------------------
        x -= pad_x
        y -= pad_y

        # -----------------------
        # SCALE BACK TO ORIGINAL IMAGE SIZE
        # -----------------------
        x1 = int((x - w / 2) / scale)
        y1 = int((y - h / 2) / scale)
        x2 = int((x + w / 2) / scale)
        y2 = int((y + h / 2) / scale)

        # -----------------------
        # DRAW BOX
        # -----------------------
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        text = f"{label} {conf:.2f}"
        cv2.putText(frame, text, (x1, y1 - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 5, (0, 255, 0), 2)

    # -----------------------
    # FPS (REAL TIME)
    # -----------------------
    end_time = time.time()
    inference_fps = 1 / (end_time - start_time + 1e-6)
    cv2.putText(frame,
                f"INF FPS: {inference_fps:.2f}",
                (20, 150),
                cv2.FONT_HERSHEY_SIMPLEX,
                5,
                (0, 0, 255),
                4)
    
    total = red_count + white_count

    if total == 0:
        state = "NO SIGNAL"

    else:
        white_ratio = white_count / total

        if white_ratio >= 0.85:
            state = "TOO HIGH (4W)"

        elif 0.60 <= white_ratio < 0.85:
            state = "SLIGHTLY HIGH"

        elif 0.40 <= white_ratio < 0.60:
            state = "ON GLIDE PATH (2R2W)"

        elif 0.15 <= white_ratio < 0.40:
            state = "SLIGHTLY LOW"

        else:
            state = "TOO LOW (4R)"
    
    history.append(state)

    counts = Counter(history)
    final_state = counts.most_common(1)[0][0]

    cv2.putText(
        frame,
        f"PAPI: {final_state}",
        (20, 300),
        cv2.FONT_HERSHEY_SIMPLEX,
        5,
        (255, 255, 0),
        4
    )
    # -----------------------
    # WRITE OUTPUT
    # -----------------------
    out.write(frame)

cap.release()
out.release()

print("DONE ✔")
print("Saved to:", output_path)

DONE ✔
Saved to: output_annotated_quantized.avi


## Testing non-quantized onnx model on video

In [3]:
# -----------------------
# CONFIG
# -----------------------
video_path = "output.avi"          # input video
model_path = r"runs\detect\train-2\weights\best.onnx"      # your YOLO26n model
output_path = "output_annotated.avi"

input_size = 640
conf_thres = 0.4

class_names = {
    0: "papi_red",
    1: "papi_white"
}

history = deque(maxlen=5)

# -----------------------
# LOAD MODEL
# -----------------------
session = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])
input_name = session.get_inputs()[0].name

# -----------------------
# VIDEO
# -----------------------
cap = cv2.VideoCapture(video_path)

frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*"XVID")
out = cv2.VideoWriter(output_path, fourcc, fps, (frame_w, frame_h))

# -----------------------
# LETTERBOX (IMPORTANT FOR YOLO)
# -----------------------
def letterbox(im, new_shape=640):
    h, w = im.shape[:2]

    scale = min(new_shape / h, new_shape / w)
    nh, nw = int(h * scale), int(w * scale)

    resized = cv2.resize(im, (nw, nh))

    top = (new_shape - nh) // 2
    bottom = new_shape - nh - top
    left = (new_shape - nw) // 2
    right = new_shape - nw - left

    padded = cv2.copyMakeBorder(
        resized, top, bottom, left, right,
        cv2.BORDER_CONSTANT, value=(114, 114, 114)
    )

    return padded, scale, left, top

# -----------------------
# PREPROCESS
# -----------------------
def preprocess(frame):
    img, scale, pad_x, pad_y = letterbox(frame, input_size)

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.astype(np.float32) / 255.0
    img = np.transpose(img, (2, 0, 1))
    img = np.expand_dims(img, axis=0)

    return img, scale, pad_x, pad_y

# -----------------------
# FPS TRACKING
# -----------------------
prev_time = time.time()

# -----------------------
# MAIN LOOP
# -----------------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    start_time = time.time()

    input_tensor, scale, pad_x, pad_y = preprocess(frame)

    outputs = session.run(None, {input_name: input_tensor})
    preds = outputs[0][0]

    red_count = 0
    white_count = 0
    # -----------------------
    # DETECTIONS
    # -----------------------
    for pred in preds:

        x, y, w, h = pred[:4]
        conf = pred[4]

        if conf < conf_thres:
            continue

        cls_id = int(pred[5]) if len(pred) > 5 else 0
        label = class_names.get(cls_id, str(cls_id))

        # COUNTING LIGHTS
        if cls_id == 0:
            red_count += 1
        elif cls_id == 1:
            white_count += 1

        # -----------------------
        # REMOVE LETTERBOX OFFSET
        # -----------------------
        x -= pad_x
        y -= pad_y

        # -----------------------
        # SCALE BACK TO ORIGINAL IMAGE SIZE
        # -----------------------
        x1 = int((x - w / 2) / scale)
        y1 = int((y - h / 2) / scale)
        x2 = int((x + w / 2) / scale)
        y2 = int((y + h / 2) / scale)

        # -----------------------
        # DRAW BOX
        # -----------------------
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        text = f"{label} {conf:.2f}"
        cv2.putText(frame, text, (x1, y1 - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 5, (0, 255, 0), 2)

    # -----------------------
    # FPS (REAL TIME)
    # -----------------------
    end_time = time.time()
    inference_fps = 1 / (end_time - start_time + 1e-6)
    cv2.putText(frame,
                f"INF FPS: {inference_fps:.2f}",
                (20, 150),
                cv2.FONT_HERSHEY_SIMPLEX,
                5,
                (0, 0, 255),
                4)
    
    total = red_count + white_count

    if total == 0:
        state = "NO SIGNAL"

    else:
        white_ratio = white_count / total

        if white_ratio >= 0.85:
            state = "TOO HIGH (4W)"

        elif 0.60 <= white_ratio < 0.85:
            state = "SLIGHTLY HIGH"

        elif 0.40 <= white_ratio < 0.60:
            state = "ON GLIDE PATH (2R2W)"

        elif 0.15 <= white_ratio < 0.40:
            state = "SLIGHTLY LOW"

        else:
            state = "TOO LOW (4R)"
    
    history.append(state)

    counts = Counter(history)
    final_state = counts.most_common(1)[0][0]

    cv2.putText(
        frame,
        f"PAPI: {final_state}",
        (20, 300),
        cv2.FONT_HERSHEY_SIMPLEX,
        5,
        (255, 255, 0),
        4
    )
    # -----------------------
    # WRITE OUTPUT
    # -----------------------
    out.write(frame)

cap.release()
out.release()

print("DONE ✔")
print("Saved to:", output_path)

DONE ✔
Saved to: output_annotated.avi


## Testing yolo26n finetuned on video

In [7]:
# -----------------------
# CONFIG
# -----------------------
video_path = "output.avi"
model_path = r"runs\detect\train-2\weights\best.pt"
output_path = "output_annotated_yolo.avi"

history = deque(maxlen=5)

# -----------------------
# LOAD MODEL (YOLO PT)
# -----------------------
model = YOLO(model_path)

# -----------------------
# VIDEO
# -----------------------
cap = cv2.VideoCapture(video_path)

frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*"XVID")
out = cv2.VideoWriter(output_path, fourcc, fps, (frame_w, frame_h))

# -----------------------
# CLASS NAMES
# -----------------------
class_names = {0: "papi_red", 1: "papi_white"}

# -----------------------
# MAIN LOOP WITH TRACKING
# -----------------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    start_time = time.time()

    # -----------------------
    # TRACKING (ByteTrack default)
    # -----------------------
    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        conf=0.4,
        verbose=False
    )

    red_count = 0
    white_count = 0

    r = results[0]

    # -----------------------
    # DRAW DETECTIONS
    # -----------------------
    if r.boxes is not None:
        for box in r.boxes:

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])
            cls_id = int(box.cls[0])
            track_id = int(box.id[0]) if box.id is not None else -1

            label = class_names.get(cls_id, str(cls_id))

            # COUNT LIGHTS
            if cls_id == 0:
                red_count += 1
            elif cls_id == 1:
                white_count += 1

            # COLOR
            color = (0, 255, 0) if cls_id == 1 else (0, 0, 255)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            text = f"ID:{track_id} {label} {conf:.2f}"
            cv2.putText(frame, text, (x1, y1 - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    # -----------------------
    # FPS
    # -----------------------
    end_time = time.time()
    inference_fps = 1 / (end_time - start_time + 1e-6)

    cv2.putText(frame,
                f"INF FPS: {inference_fps:.2f}",
                (20, 150),
                cv2.FONT_HERSHEY_SIMPLEX,
                3,
                (0, 0, 255),
                2)

    # -----------------------
    # PAPI LOGIC
    # -----------------------
    total = red_count + white_count

    if total == 0:
        state = "NO SIGNAL"

    else:
        white_ratio = white_count / total

        if white_ratio >= 0.85:
            state = "TOO HIGH (4W)"
        elif 0.60 <= white_ratio < 0.85:
            state = "SLIGHTLY HIGH"
        elif 0.40 <= white_ratio < 0.60:
            state = "ON GLIDE PATH (2R2W)"
        elif 0.15 <= white_ratio < 0.40:
            state = "SLIGHTLY LOW"
        else:
            state = "TOO LOW (4R)"

    # -----------------------
    # SMOOTHING
    # -----------------------
    history.append(state)
    final_state = Counter(history).most_common(1)[0][0]

    cv2.putText(frame,
                f"PAPI: {final_state}",
                (20, 300),
                cv2.FONT_HERSHEY_SIMPLEX,
                4,
                (255, 255, 0),
                4)

    # -----------------------
    # WRITE FRAME
    # -----------------------
    out.write(frame)

cap.release()
out.release()

print("DONE ✔")
print("Saved to:", output_path)

DONE ✔
Saved to: output_annotated_yolo.avi
